# Урок 6.2. Данные для дообучения и обучение на Colab

В уроке 6.1 мы решили: фирменный стиль «Пингвин.Хост» — задача для LoRA, r=16,
QLoRA на бесплатном Colab. Сегодня — вся дорога до обученного адаптера:

🎯 **Цели урока:**

- Превратить 45 тикетов в обучающий датасет: chat-формат, ChatML, train/val.
- Посчитать примеры в токенах и выбрать `max_seq_length` не наугад.
- Понять каждый гиперпараметр в конфиге обучения, а не копировать их «как у всех».
- Обучить адаптер на бесплатном Colab (T4) ноутбуком `colab-train-lora.ipynb`.
- Знать, как читать loss и как выглядит переобучение на 45 примерах.

**Что понадобится:** локально — только Ollama (для подсчёта токенов); для обучения —
Google-аккаунт и 15 минут бесплатного Colab. Свой GPU не обязателен.

## 1. Из тикетов — в диалоги

В `train.jsonl` лежит alpaca-подобный формат: `instruction` / `input` / `output`.
Обучать мы будем **чат-модель**, а чат-модели едят диалоги — списки сообщений
с ролями (как `/api/chat` из модуля 1). Конвертируем.

Одно важное решение сразу: **обучаем без system-промпта**. Мы хотим, чтобы стиль
жил в весах: пользователь спрашивает — модель отвечает пингвином, без напоминаний.
Альтернатива (вшить в каждый пример короткий system «Ты — поддержка Пингвин.Хост»)
тоже рабочая: стиль тогда «включается» промптом, а без него модель ближе
к базовой. Это трейдофф между «стиль всегда» и «стилем можно управлять». Для
нашего бота поддержки — всегда.

In [ ]:
import json
import random
from pathlib import Path

DATA_DIR = Path("data")
train_raw = [json.loads(line) for line in
             (DATA_DIR / "train.jsonl").read_text(encoding="utf-8").splitlines()]

def to_messages(example: dict) -> dict:
    """alpaca-формат -> чат без system: стиль должен жить в весах."""
    question = example["instruction"]
    if example.get("input"):                       # у нас поле пустое, но формат общий
        question += "\n\n" + example["input"]
    return {"messages": [
        {"role": "user", "content": question},
        {"role": "assistant", "content": example["output"]},
    ]}

dataset = [to_messages(ex) for ex in train_raw]
print(f"Диалогов: {len(dataset)}")
print(json.dumps(dataset[0], ensure_ascii=False, indent=1)[:400], "...")

## 2. ChatML: что модель увидит на самом деле

Список сообщений — это удобство для нас. Перед обучением (и перед каждым
инференсом) он разворачивается в **шаблон чата** — для qwen это ChatML,
который мы разбирали руками в уроке 1.6:

```
<|im_start|>user
Мой сайт отдаёт ошибку 502 Bad Gateway, что делать?<|im_end|>
<|im_start|>assistant
Здравствуйте! Поддержка «Пингвин.Хост» на связи. 🐧
...<|im_end|>
```

Модель учится продолжать ровно такую последовательность. Отсюда два следствия,
из-за которых ломается больше дообучений, чем из-за любых гиперпараметров:

1. **Шаблон должен быть родной.** Обучите qwen на llama-шаблоне — получите модель,
   которая на проде (где Ollama применит родной ChatML) видит «чужую» разметку.
   В unsloth шаблон берётся из токенизатора базовой модели — не переопределяйте его.
2. **Loss нужно считать только на ответе.** В последовательности есть и вопрос
   пользователя. Если учить на всём подряд, модель прилежно учится... писать
   вопросы клиентов. Для этого в ноутбуке обучения будет `train_on_responses_only`:
   токены вопроса маскируются из loss, штраф начисляется только за ответ.

In [ ]:
def render_chatml(messages: list[dict]) -> str:
    """ChatML руками - только чтобы УВИДЕТЬ, чему учим (в обучении это сделает токенизатор)."""
    parts = [f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>" for m in messages]
    return "\n".join(parts)

sample = render_chatml(dataset[0]["messages"])
print(sample[:350])
print(f"\n... всего {len(sample)} символов")

## 3. Сколько это в токенах

`max_seq_length` — до какой длины обучающие примеры влезают целиком (что длиннее —
молча обрежется, и модель выучит ответы без финалки!). Ставить наугад 4096 —
платить памятью за воздух. Посчитаем честно.

Токенизатора qwen у нас локально нет (и качать не будем) — но токены умеет
считать сама Ollama: `prompt_eval_count` в ответе `/api/chat` (телеметрия
из урока 5.5). Маленькая хитрость: чтобы ответ ассистента тоже попал в *промпт*
(и в счётчик), добавим в конец диалога фиктивный ход пользователя.

In [ ]:
import httpx

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "qwen2.5:3b"

def count_tokens(messages: list[dict]) -> int:
    """Токены диалога по счётчику Ollama (вместе с разметкой ChatML)."""
    probe = messages + [{"role": "user", "content": "x"}]  # чтобы assistant вошёл в промпт
    response = httpx.post(f"{OLLAMA_URL}/api/chat", json={
        "model": MODEL, "messages": probe, "stream": False,
        "options": {"num_predict": 1},
    }, timeout=120)
    response.raise_for_status()
    return response.json().get("prompt_eval_count", 0)

try:
    lengths = sorted(count_tokens(d["messages"]) for d in dataset)
    n = len(lengths)
    print(f"Токенов в примере: min {lengths[0]}, медиана {lengths[n // 2]}, "
          f"p95 {lengths[int(n * 0.95)]}, max {lengths[-1]}")
    bins = {}
    for length in lengths:
        bins[length // 50 * 50] = bins.get(length // 50 * 50, 0) + 1
    for start in sorted(bins):
        print(f"  {start:>4}-{start + 49:<4} {'#' * bins[start]}")
except httpx.HTTPError as exc:
    print(f"[пропущено: Ollama недоступна - {exc.__class__.__name__}; "
          "по прошлому замеру max ~281 токена]")

В моём прогоне: медиана **243 токена**, максимум **285** (вместе с разметкой;
у вас числа могут отличаться на единицы из-за фиктивного хода). Значит,
`max_seq_length = 512` покрывает весь датасет с почти двукратным запасом — а
память на активации при обучении растёт с длиной, так что запас без фанатизма.

Заодно это смета стоимости: 45 примеров × ~240 токенов в медиане × 3 эпохи ≈
32 тысячи токенов на всё обучение. Для сравнения: few-shot из урока 6.1 сжигает
660 токенов **на каждый запрос** — дообучение окупается за полсотни обращений.

## 4. Train / val: 40 + 5

Даже на 45 примерах нужен валидационный кусочек — не для метрик качества
(5 примеров — не статистика), а как **датчик переобучения**: если train-loss
падает, а val-loss пополз вверх — модель начала зазубривать.

Полноценный экзамен у нас другой: `test_questions.jsonl` — 6 вопросов, которых
модель не увидит вообще, и `style_score` из урока 6.1. Это разделение ролей
как в уроке 3.3: loss — для процесса обучения, свой eval — для приёмки.

In [ ]:
random.seed(42)                      # воспроизводимый сплит
shuffled = dataset.copy()
random.shuffle(shuffled)
val, train = shuffled[:5], shuffled[5:]

def save_jsonl(rows: list[dict], path: Path) -> None:
    path.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in rows) + "\n",
                    encoding="utf-8")

save_jsonl(train, DATA_DIR / "train_messages.jsonl")
save_jsonl(val, DATA_DIR / "val_messages.jsonl")
print(f"train: {len(train)} диалогов -> data/train_messages.jsonl")
print(f"val:   {len(val)} диалогов -> data/val_messages.jsonl")
print("отложенный экзамен: data/test_questions.jsonl (в обучение НЕ попадает)")

## 5. Гиперпараметры — по одному и зачем

Конфиг обучения из ноутбука `colab-train-lora.ipynb`, построчно:

| Параметр | Значение | Почему |
|---|---|---|
| `r` | 16 | посчитали в 6.1: для стиля хватает, на 45 примерах больший ранг = зазубривание |
| `lora_alpha` | 16 | масштаб alpha/r = 1, стандартный старт |
| `lora_dropout` | 0 | датасет крошечный, каждая крупица данных на счету; от переобучения защищаемся эпохами |
| `target_modules` | все linear (q,k,v,o + gate,up,down) | дефолт unsloth; ~30 млн параметров |
| `max_seq_length` | 512 | замерили: максимум ~280 токенов, двукратный запас |
| `learning_rate` | 2e-4 | норма для LoRA (в 100 раз выше, чем для полного FT: учим-то маленькие матрицы с нуля) |
| `num_train_epochs` | 3 | 45 примеров: 1 эпоха — не доучится, 10 — зазубрит |
| `per_device_train_batch_size` | 4 | T4 позволяет; 40 примеров / 4 = 10 шагов на эпоху |
| `gradient_accumulation_steps` | 2 | эффективный батч 8 — стабильнее градиенты на мелком датасете |
| `warmup_steps` | 3 | первые шаги — с малым lr, чтобы нулевая B не получила «пинок» |
| `optim` | adamw_8bit | Adam-статистики в 8 бит: ещё минус память (дух QLoRA) |
| `seed` | 42 | воспроизводимость (насколько она вообще бывает на GPU — урок 3.3) |

Итого: 3 эпохи × 5 шагов (40 примеров / эффективный батч 8) = **15 шагов
обучения**. Да, всего пятнадцать. Стиль — простая задача.

## 6. Обучение: ноутбук для Colab

Обучение живёт в отдельном ноутбуке **`colab-train-lora.ipynb`** (лежит рядом
в папке модуля). Он самодостаточный: его единственная зависимость — два jsonl-файла,
которые вы только что сохранили.

Пошагово:

1. Откройте [colab.research.google.com](https://colab.research.google.com) →
   «Загрузить» → выберите `colab-train-lora.ipynb` из папки модуля.
2. **Среда выполнения → Сменить среду выполнения → T4 GPU.** Без этого шага
   unsloth откажется стартовать (и правильно сделает).
3. Запускайте ячейки по порядку. Вторая попросит загрузить
   `train_messages.jsonl` и `val_messages.jsonl` из `data/`.
4. Обучение — 15 шагов, на T4 это ~2–4 минуты. Установка библиотек и скачивание
   базовой модели дольше — суммарно закладывайте 15–20 минут.
5. Последняя ячейка соберёт и скачает два артефакта:
   - `pingvin-lora.zip` — адаптер (десятки МБ) — **страховочная копия**;
   - `pingvin-q4_k_m.gguf` — модель уже со вшитым стилем, готовая для Ollama (~2 ГБ).
6. Положите артефакты в папку модуля `06-fine-tuning-lora/artifacts/` — урок 6.3
   заберёт их оттуда.

Честное предупреждение: ноутбук написан по документации unsloth и стандартному
API trl; эти библиотеки развиваются быстро, и через полгода какая-нибудь ячейка
может попросить другое имя аргумента. На этот случай в конце ноутбука есть раздел
«Если сломалось» — а принципы (шаблон чата, loss только на ответе, малый r) не
устаревают.

## 7. Ключевые ячейки Colab-ноутбука — разбор

Пока Colab устанавливает пакеты, разберём, что он будет делать.

**Загрузка базы в 4 битах** — QLoRA из урока 6.1 в одну строку:

```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",  # база уже в NF4
    max_seq_length=512,
    load_in_4bit=True,
)
```

Обратите внимание на имя: **Instruct**, а не base. Мы дообучаем модель, которая
уже умеет вести диалог (урок 1.6 про пропасть между base и instruct), — учим
только стиль, а не «чат с нуля».

**Навешивание адаптеров** — та самая арифметика из 6.1:

```python
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",   # экономия памяти на активациях
    random_state=42,
)
```

**Данные:** каждый диалог прогоняется через родной шаблон токенизатора —
никакого самодельного ChatML в обучении:

```python
def to_text(row):
    return {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False)}
```

**Loss только на ответах** — маскируем вопросы пользователя:

```python
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
```

Строки-маркеры — это куски ChatML из раздела 2: по ним библиотека находит,
где в последовательности вопрос, а где ответ.

**Сохранение двух артефактов:**

```python
model.save_pretrained("pingvin-lora")            # адаптер: A, B и конфиг
tokenizer.save_pretrained("pingvin-lora")
model.save_pretrained_gguf("pingvin-gguf",       # merge + конвертация + Q4_K_M
                           tokenizer, quantization_method="q4_k_m")
```

`save_pretrained_gguf` делает за нас весь маршрут урока 6.3 (merge → GGUF →
квантизация) прямо на Colab. Адаптер отдельно мы всё равно сохраняем: он
крошечный, а из него всегда можно пересобрать модель — в том числе под другую
квантизацию.

## 8. Как читать логи обучения

На 15 шагах loss-кривая выглядит примерно так (числа — ориентир, не эталон):

```
шаг  1-3   loss ~2.3-1.8   warmup: модель впервые видит наш формат
шаг  4-8   loss ~1.5-0.8   основное падение: выучивает скелет ответа
шаг  9-15  loss ~0.7-0.4   полировка: формулировки, финалка
```

На что смотреть:

- **Loss не падает вовсе** (~2+ до конца) → чаще всего сломан шаблон чата или
  маскирование ответов: модель учится не тому. Проверьте печать первого
  обучающего примера — ноутбук выводит его перед стартом.
- **Loss упал к ~0.1–0.2** → на 45 примерах это почти наверняка зазубривание.
  Ждите на приёмке в 6.3 дословных кусков train-ответов на чужие вопросы.
- **eval_loss (на наших 5 val-диалогах) пополз вверх, пока train падает** →
  классическое переобучение; для нашего масштаба лечится «меньше эпох».

И главное: низкий loss — это **не** приёмка. Приёмка — `style_score` на отложенных
вопросах и проверка общих навыков (урок 6.3). Loss лишь говорит, что обучение шло.

## ⚠️ Частые ошибки

1. **Чужой шаблон чата.** Обучили с llama-разметкой, инференс в Ollama — с ChatML:
   стиль «не пришился». Шаблон должен совпадать в обучении и на проде.
2. **Loss на всей последовательности** (забыли `train_on_responses_only`) —
   модель учится писать вопросы клиентов и разбавляет этим стиль ответов.
3. **`max_seq_length` меньше длинного примера.** Обрезка молчаливая: модель
   никогда не увидит финалку длинных ответов — и перестанет её ставить.
4. **Дообучение base-модели вместо instruct.** База не умеет диалог (урок 1.6);
   45 примеров не научат её чату — только испортят генерацию.
5. **Закрыли Colab, не скачав артефакты.** Среда эфемерна: слетела сессия —
   обучайте заново. Скачивание — не «потом», а последняя ячейка ноутбука.
6. **Сравнение «до/после» на train-вопросах.** На них после обучения будет
   красиво даже при полном зазубривании. Только отложенные вопросы.

## Упражнения

1. **Расширьте датасет.** Напишите 3 своих примера в фирменном стиле (новые темы:
   например, «как включить двухфакторную аутентификацию», «сайт в чёрном списке
   Роскомнадзора», «нужна помощь с .htaccess»). Прогоните их через `style_score` —
   все ли якоря на месте? Добавьте в `train.jsonl` и пересоберите
   `train_messages.jsonl`.

2. **Арифметика шагов.** Заказчик принёс ещё 155 примеров (итого 200). Сколько
   шагов обучения получится при том же конфиге (batch 4, grad_accum 2, 3 эпохи)?
   А сколько эпох разумно на 200 примерах?

3. **Спроектируйте датасет** для задачи б) из урока 6.1 (ответы строго в JSON
   `{"answer": ..., "confidence": ...}`): какие поля, сколько примеров, что пойдёт
   в user-ход, что в assistant-ход, и как выглядит style_score для такого формата.

In [ ]:
# Решение 2: арифметика шагов
examples, batch, accum, epochs = 200, 4, 2, 3
steps_per_epoch = examples // (batch * accum)
print(f"Шагов на эпоху: {examples} / {batch * accum} = {steps_per_epoch}")
print(f"Всего: {steps_per_epoch * epochs} шагов")
# На 200 примерах обычно хватает 1-2 эпох: данных больше -> повторять их нужно
# меньше. Правило: следите за val_loss, а не за красивым числом эпох.

**Решение 1** — главное, что проверяет чеклист: приветствие дословно, 🐧 на месте,
«Что происходит:» одной строкой диагноза, «Что делать:» с нумерацией в терминах
панели, финалка дословно. Дословность якорей важнее содержания шагов: стиль
учится с точностью до символа.

**Решение 3 (эскиз).** user — произвольный вопрос; assistant — **только** валидный
JSON без прозы вокруг: `{"answer": "...", "confidence": 0.9}`. Примеров 50+,
с разбросом уверенности (не все 0.9!) и с примерами отказа
(`{"answer": null, "confidence": 0.0}` на вопросы не по теме — иначе бот будет
уверенно отвечать на всё). style_score здесь — это `json.loads` + проверка схемы
(рифма с чекерами урока 5.2), а на проде добавится грамматика из урока 5.4:
дообучение повышает вероятность правильного формата, грамматика делает её единицей.

## Мини-квиз

**1. Зачем `train_on_responses_only`, если и вопрос, и ответ — часть нашего датасета?**

<details><summary>Ответ</summary>

Датасет учит модель *отвечать*, а не *спрашивать*. Без маскирования loss начисляется
и за токены вопроса — модель тратит ёмкость на имитацию клиентов, а стиль ответов
размывается. Вопросы остаются в контексте (модель их видит), но штраф — только за ответ.
</details>

**2. Почему lr для LoRA (2e-4) на два порядка выше, чем для полного дообучения (~2e-6)?**

<details><summary>Ответ</summary>

Мы учим не гигантские предобученные веса (их легко «сломать» большим шагом), а
маленькие свежие матрицы A и B, стартующие с нуля. Им нужно быстро вырасти до
рабочих значений — консервативный lr полного FT их просто не разбудит за 15 шагов.
</details>

**3. Обучение закончилось с loss 0.05. Радоваться?**

<details><summary>Ответ</summary>

Насторожиться: на 45 примерах такой loss — почти наверняка зазубривание датасета
наизусть. Проверка — отложенные вопросы: если в ответах дословные куски чужих
train-ответов, откатывайтесь на меньше эпох / меньший r.
</details>

**4. Чем плох `max_seq_length=4096` «на всякий случай» при наших 280 токенах максимум?**

<details><summary>Ответ</summary>

Память на активации и время шага растут с длиной последовательности — вы платите
за воздух, на T4 это может стоить OOM. Замерили максимум — взяли с razumным
запасом (512), как num_ctx в уроке 5.6.
</details>

**5. Val-набор из 5 примеров — это же не статистика. Зачем он?**

<details><summary>Ответ</summary>

Как датчик направления, а не как метрика: train-loss падает при любом раскладе,
а вот если val-loss развернулся вверх — модель перестала обобщать и начала
зазубривать. Для оценки качества есть отдельный экзамен — style_score на
отложенных вопросах.
</details>

## 🎓 Итоги

- Датасет для чат-модели — это диалоги: мы обучаем **без system-промпта**, чтобы
  стиль жил в весах, и с loss **только на ответах** (`train_on_responses_only`).
- Шаблон чата (ChatML для qwen — урок 1.6) обязан совпадать в обучении и на проде.
- Токены посчитали счётчиком Ollama: максимум ~280 → `max_seq_length=512`.
- Сплит 40/5: val — датчик переобучения; настоящий экзамен — 6 отложенных
  вопросов + `style_score`.
- Конфиг: r=16, alpha=16, lr 2e-4, 3 эпохи, эффективный батч 8 — всего 15 шагов
  на бесплатном T4.
- Артефакты обучения: адаптер (десятки МБ, страховка) и готовый GGUF Q4_K_M.

**Дальше — урок 6.3:** возвращаем GGUF домой — Modelfile, `ollama create`,
и приёмка: стиль на отложенных вопросах, проверка общих навыков и что делать,
если приёмка провалена.

## 📚 Что почитать

- [Unsloth: Fine-tuning Guide](https://docs.unsloth.ai/) — актуальные ноутбуки под разные модели
- [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) — что происходит под капотом обучения
- [Qwen2.5: шаблоны чата](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct) — карточка базовой модели
